# Praktikum Minggu 4 — Object Recognition & Dasar CNN (Lembar Kerja Mahasiswa)
**Mata Kuliah Computer Vision** · Universitas Sanata Dharma

| | |
|---|---|
| **Nama** | Leonardus Dale Masan |
| **NIM** |235314071 |
| **Kelas** |Komputer Pelihat B |

**Isi praktikum**
1. Konvolusi, pooling, dan rumus ukuran output ditulis manual dengan NumPy
2. Verifikasi jawaban Bagian B lembar latihan
3. Membangun `SmallCNN` dan menelusuri shape serta jumlah parameter
4. Memuat CIFAR-10 langsung (siap pakai)
5. Melatih dan membandingkan tiga varian: MaxPool, AvgPool, Global Average Pooling
6. Visualisasi filter, feature map, dan akurasi per kelas

**Petunjuk pengerjaan**
- Isi setiap bagian bertanda `# TODO`. Hapus baris `raise NotImplementedError` setelah kode Anda selesai.
- Sel **cek otomatis** akan menampilkan ✔ bila jawaban benar. Kunci jawaban tidak ditampilkan; yang dicek hanya sidik (hash) hasil Anda.
- Jangan mengubah nama fungsi atau kelas, karena sel-sel berikutnya bergantung padanya.
- Kumpulkan notebook (.ipynb) yang sudah dijalankan penuh, termasuk jawaban pertanyaan analisis di bagian akhir.

In [ ]:
# ============================================================
# 0. Setup: import, seed, device, konfigurasi
# ============================================================
import os, time, random, pickle, tarfile, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {DEVICE}')

EPOCHS = int(os.environ.get('CV_EPOCHS', 5))
BATCH_SIZE = 64
LR = 1e-3

In [ ]:
# Fungsi cek otomatis — JANGAN DIUBAH
_KUNCI = {'B1': '4f7c13d7c1c7', 'B1_relu': 'e0b612410c05', 'B2_max': '22ec042ab416', 'B2_avg': '02550701e335', 'B3': 'fa8b0ac29cfa', 'B4': 'afcf2406b725', 'B5': '7f5d0d1b9107', 'B6': '359e34c3951a', 'GAP': '9b013ed23e43'}

def _sidik(v):
    return hashlib.sha256(str(np.round(np.asarray(v, dtype=float), 2).tolist()).encode()).hexdigest()[:12]

def cek(nama, nilai):
    '''Bandingkan sidik jawaban Anda dengan kunci tanpa membuka kuncinya.'''
    ok = _sidik(nilai) == _KUNCI[nama]
    print(f"{'✔' if ok else '✘'} {nama}: {'benar' if ok else 'belum tepat, periksa lagi langkah Anda'}")
    return ok

## 1. Konvolusi & Pooling Manual (NumPy)

Di deep learning, "konvolusi" sebenarnya adalah **cross-correlation**: kernel tidak dibalik.

$$Y[i,j] = \sum_m \sum_n X[i\cdot S + m,\; j\cdot S + n]\,K[m,n] + b \qquad O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

Jumlah parameter conv layer: $(K \cdot K \cdot C_{in} + 1)\cdot C_{out}$

In [ ]:
def conv2d_manual(X, K, stride=1, padding=0, bias=0.0):
    '''Cross-correlation 2D satu channel (setara nn.Conv2d tanpa batch/channel).

    X : (H, W) input, K : (k, k) kernel. Mengembalikan array (H_out, W_out).
    '''
    X = np.pad(np.asarray(X, dtype=float), padding)
    k = K.shape[0]
    H_out = (X.shape[0] - k) // stride + 1
    W_out = (X.shape[1] - k) // stride + 1
    Y = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            # TODO 1a: ambil jendela X berukuran k x k yang dimulai di baris i*stride, kolom j*stride,
            #          kalikan elemen demi elemen dengan K, jumlahkan, lalu tambahkan bias.
            raise NotImplementedError('TODO 1a')
    return Y


def pool2d_manual(X, size=2, stride=2, mode='max'):
    '''Max/average pooling 2D satu channel.'''
    X = np.asarray(X, dtype=float)
    # TODO 1b: hitung H_out dan W_out, lalu isi array output dengan np.max atau np.mean dari setiap jendela.
    raise NotImplementedError('TODO 1b')


def out_size(W, K, P=0, S=1):
    '''Ukuran output conv/pooling: floor((W - K + 2P)/S) + 1.'''
    # TODO 1c
    raise NotImplementedError('TODO 1c')


def conv_params(K, C_in, C_out, bias=True):
    '''Jumlah parameter conv layer: (K*K*C_in + bias) * C_out.'''
    # TODO 1d
    raise NotImplementedError('TODO 1d')

In [ ]:
# Cek dengan contoh slide 9, 10, dan 13 (angkanya ada di slide)
X_slide = np.array([[1,2,0,1,3],[0,1,2,3,1],[1,0,1,2,0],[2,1,0,1,1],[0,2,1,0,2]])
K_vert  = np.array([[1,0,-1],[1,0,-1],[1,0,-1]])
Y_np = conv2d_manual(X_slide, K_vert)
Y_pt = F.conv2d(torch.tensor(X_slide, dtype=torch.float32)[None, None],
                torch.tensor(K_vert, dtype=torch.float32)[None, None]).squeeze().numpy()
print('Manual :\n', Y_np, '\nPyTorch:\n', Y_pt)
assert np.allclose(Y_np, Y_pt), 'Konvolusi manual belum sama dengan PyTorch'

P_slide = np.array([[1,3,2,1],[4,6,5,0],[3,1,1,2],[0,2,7,4]])
assert pool2d_manual(P_slide, mode='max').tolist() == [[6, 5], [3, 7]], 'Max pooling belum tepat'
assert pool2d_manual(P_slide, mode='avg').tolist() == [[3.5, 2.0], [1.5, 3.5]], 'Average pooling belum tepat'
assert [out_size(*a) for a in [(5,3,0,1), (32,3,1,1), (32,5,0,1), (224,7,3,2), (227,11,0,4)]] == [3, 32, 28, 112, 55]
assert conv_params(3, 3, 64) == 1792
print('✔ Bagian 1 lulus semua pengecekan')

### Efek kernel pada citra nyata
Kernel yang pada Minggu 3 dirancang manual adalah jenis pola yang nanti **dipelajari sendiri** oleh conv1.

In [ ]:
from skimage import data as skdata      # citra contoh bawaan scikit-image (tidak perlu unduh)
img = skdata.camera().astype(float) / 255.0
kernels = {'Tepi vertikal (Prewitt)': K_vert,
           'Sobel horizontal': np.array([[1,2,1],[0,0,0],[-1,-2,-1]]),
           'Laplacian': np.array([[0,1,0],[1,-4,1],[0,1,0]])}
img_t = torch.tensor(img, dtype=torch.float32)[None, None]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Asli')
for ax, (name, k) in zip(axes[1:], kernels.items()):
    out = F.conv2d(img_t, torch.tensor(k, dtype=torch.float32)[None, None], padding=1)
    ax.imshow(out.squeeze().abs().numpy(), cmap='magma'); ax.set_title(name)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Verifikasi Jawaban Lembar Latihan (Bagian B)
Gunakan fungsi dari bagian 1 atau modul PyTorch. Setelah variabel terisi, fungsi `cek()` akan memberi tahu apakah hasil Anda benar.

In [ ]:
# B1 — konvolusi Laplacian + ReLU
X_b1 = np.array([[2,0,1,3],[1,3,0,1],[0,1,2,2],[3,1,0,1]])
K_lap = np.array([[0,1,0],[1,-4,1],[0,1,0]])
Y_b1 = None        # TODO 2a: hasil konvolusi X_b1 dengan K_lap
relu_b1 = None     # TODO 2b: ReLU dari Y_b1

# B2 — pooling 2x2 stride 2
F_b2 = np.array([[5,1,0,2],[3,8,4,4],[1,0,6,2],[2,9,3,1]])
max_b2 = None      # TODO 2c
avg_b2 = None      # TODO 2d

# B3 — conv1 AlexNet: input 227, kernel 11, stride 4, padding 0, 96 filter; lalu max pool 3x3 stride 2
o = None           # TODO 2e: ukuran output conv1
p = None           # TODO 2f: jumlah parameter conv1
o2 = None          # TODO 2g: ukuran output setelah pooling

# B5 — tiga conv 3x3 vs satu conv 7x7, C = 64 channel masuk & keluar, TANPA bias
C = 64
p3 = None          # TODO 2h
p7 = None          # TODO 2i

# B6 — skor z = [3, 1, 0], label benar = kelas kedua (indeks 1). Gunakan F.softmax dan F.cross_entropy.
z = torch.tensor([[3.0, 1.0, 0.0]])
prob = None        # TODO 2j
loss_b6 = None     # TODO 2k: nilai loss sebagai float (pakai .item())

In [ ]:
# Cek otomatis Bagian B
cek('B1', Y_b1); cek('B1_relu', relu_b1)
cek('B2_max', max_b2); cek('B2_avg', avg_b2)
cek('B3', (o, p, o2)); cek('B5', (p3, p7)); cek('B6', loss_b6)

In [ ]:
# B4 — bangun jaringan bergaya LeNet untuk MNIST 28x28x1:
# Conv 5x5 (6 filter) -> ReLU -> MaxPool 2 -> Conv 5x5 (16 filter) -> ReLU -> MaxPool 2
# -> Flatten -> Linear(?, 120) -> ReLU -> Linear(120, 10)
lenet = nn.Sequential(
    # TODO 2l: lengkapi layer-layer di sini. Tentukan sendiri ukuran input Linear pertama.
)
x_dummy = torch.zeros(1, 1, 28, 28)
print('Output shape:', lenet(x_dummy).shape)          # harus [1, 10]
n_lenet = sum(p.numel() for p in lenet.parameters())
print(f'Total parameter = {n_lenet:,}')
cek('B4', n_lenet)

## 3. `SmallCNN`

Arsitektur mengikuti slide 16 & 18. Argumen `pool` dan `head` membentuk tiga varian eksperimen:

| Varian | Pooling | Classifier |
|---|---|---|
| `max_fc` | MaxPool | Flatten → FC 128 → ReLU → FC n_cls |
| `avg_fc` | AvgPool | Flatten → FC 128 → ReLU → FC n_cls |
| `max_gap` | MaxPool | Global Average Pooling → FC n_cls |

In [ ]:
class SmallCNN(nn.Module):
    '''CNN dua blok konvolusi untuk citra 32x32x3.

    n_cls : jumlah kelas, pool : 'max' | 'avg', head : 'fc' | 'gap'
    '''
    def __init__(self, n_cls=10, pool='max', head='fc'):
        super().__init__()
        Pool = nn.MaxPool2d if pool == 'max' else nn.AvgPool2d
        # TODO 3a: feature extractor
        #   Conv2d 3->16 (kernel 3, padding 1) -> ReLU -> Pool(2)
        #   Conv2d 16->32 (kernel 3, padding 1) -> ReLU -> Pool(2)
        self.features = nn.Sequential(
        )
        if head == 'fc':
            # TODO 3b: Flatten -> Linear(?, 128) -> ReLU -> Linear(128, n_cls)
            self.classifier = nn.Sequential(
            )
        else:
            # TODO 3c: Global Average Pooling (nn.AdaptiveAvgPool2d(1)) -> Flatten -> Linear(?, n_cls)
            self.classifier = nn.Sequential(
            )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
def trace_model(model, input_shape=(1, 3, 32, 32)):
    '''Cetak shape output dan jumlah parameter setiap layer daun.'''
    rows, hooks = [], []
    def hook(mod, inp, out):
        rows.append({'Layer': mod.__class__.__name__, 'Output shape': tuple(out.shape[1:]),
                     'Parameter': sum(p.numel() for p in mod.parameters(recurse=False))})
    for m in model.modules():
        if len(list(m.children())) == 0:
            hooks.append(m.register_forward_hook(hook))
    model.eval()
    with torch.no_grad():
        model(torch.zeros(input_shape))
    for h in hooks: h.remove()
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print(f"{'TOTAL':>40}: {df['Parameter'].sum():,}")
    return df


df_trace = trace_model(SmallCNN())
assert df_trace['Parameter'].sum() == 268650, 'Jumlah parameter belum sama dengan slide 16 (268.650)'
print('✔ SmallCNN sesuai slide 16')

In [ ]:
VARIANTS = {'max_fc': dict(pool='max', head='fc'),
            'avg_fc': dict(pool='avg', head='fc'),
            'max_gap': dict(pool='max', head='gap')}
for name, cfg in VARIANTS.items():
    n = sum(p.numel() for p in SmallCNN(**cfg).parameters())
    print(f'{name:8s}: {n:>8,} parameter')
cek('GAP', sum(p.numel() for p in SmallCNN(head='gap').parameters()))

## 4. Data CIFAR-10 (siap pakai)

CIFAR-10 berisi 60.000 citra berwarna 32×32 dalam 10 kelas: 50.000 citra latih dan 10.000 citra uji (Krizhevsky, 2009). Sel di bawah mengunduh dataset langsung dari server resmi melalui `torchvision` (±163 MB, biasanya kurang dari 1 menit di Colab). Tidak ada yang perlu disiapkan secara manual.

**Tips hemat waktu:** runtime Colab dihapus setiap kali sesi berakhir. Set `SIMPAN_DI_DRIVE = True` agar dataset disimpan di Google Drive, sehingga di sesi berikutnya tidak perlu diunduh ulang (`torchvision` otomatis melewati unduhan bila berkas sudah ada).

In [ ]:
# ===== KONFIGURASI DATA =====
SIMPAN_DI_DRIVE = False          # True: simpan dataset di Google Drive agar tidak unduh ulang tiap sesi

DATA_DIR = './data'
if SIMPAN_DI_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA_DIR = '/content/drive/MyDrive/CV/data'
    except ImportError:
        print('Bukan di Colab — dataset disimpan di ./data')
os.makedirs(DATA_DIR, exist_ok=True)

MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)   # statistik data latih CIFAR-10
tf_train = T.Compose([T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MEAN, STD)])  # augmentasi ringan
tf_test  = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = torchvision.datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=tf_train)
test_ds  = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=tf_test)
CLASSES, N_CLS = train_ds.classes, len(train_ds.classes)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=256,        shuffle=False, num_workers=2)
print(f'Kelas ({N_CLS}): {CLASSES}')
print(f'Latih: {len(train_ds):,} | Uji: {len(test_ds):,} | lokasi: {DATA_DIR}')

In [ ]:
# Distribusi kelas: CIFAR-10 seimbang (5.000 latih dan 1.000 uji per kelas)
counts = pd.Series(train_ds.targets).value_counts().sort_index()
counts.index = CLASSES
print(counts.to_string())

In [ ]:
# Contoh citra uji (didenormalisasi agar warna tampil benar)
imgs, labels = next(iter(DataLoader(test_ds, batch_size=10, shuffle=True)))
imgs = imgs * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
fig, axes = plt.subplots(1, len(imgs), figsize=(15, 2))
for ax, im, lb in zip(axes, imgs, labels):
    ax.imshow(im.permute(1, 2, 0).clamp(0, 1)); ax.set_title(CLASSES[lb], fontsize=9); ax.axis('off')
plt.show()

## 5. Fungsi Pelatihan & Evaluasi

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    '''Satu epoch: forward -> loss -> backward -> update. Mengembalikan (loss, akurasi).'''
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        # TODO 5: lima langkah inti pelatihan
        #   1) nolkan gradien optimizer
        #   2) forward: hitung logits = model(x)
        #   3) hitung loss dengan criterion
        #   4) backward
        #   5) update bobot dengan optimizer
        raise NotImplementedError('TODO 5')
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    '''Evaluasi tanpa gradien. Mengembalikan (loss, akurasi, prediksi, label).'''
    model.eval()
    total_loss, preds, gts = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        total_loss += criterion(logits, y).item() * x.size(0)
        preds.append(logits.argmax(1).cpu()); gts.append(y.cpu())
    preds, gts = torch.cat(preds), torch.cat(gts)
    return total_loss / len(gts), (preds == gts).float().mean().item(), preds, gts


def run_experiment(name, cfg, epochs=EPOCHS):
    '''Latih satu varian SmallCNN dan simpan riwayat metrik.'''
    torch.manual_seed(SEED)                         # inisialisasi sama untuk semua varian
    model = SmallCNN(n_cls=N_CLS, **cfg).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    hist = {k: [] for k in ('train_loss', 'train_acc', 'test_loss', 'test_acc')}
    t0 = time.time()
    for ep in range(1, epochs + 1):
        tl, ta = train_one_epoch(model, train_dl, opt, crit)
        vl, va, _, _ = evaluate(model, test_dl, crit)
        for k, v in zip(hist, (tl, ta, vl, va)): hist[k].append(v)
        print(f'[{name}] epoch {ep}/{epochs}  train loss {tl:.3f} acc {ta:.3f} | test loss {vl:.3f} acc {va:.3f}')
    hist['time_s'] = time.time() - t0
    return model, hist

## 6. Eksperimen Tiga Varian

In [ ]:
results, models = {}, {}
for name, cfg in VARIANTS.items():
    models[name], results[name] = run_experiment(name, cfg)
    print('-' * 70)

In [ ]:
summary = pd.DataFrame([{
    'Varian': name,
    'Parameter': sum(p.numel() for p in models[name].parameters()),
    'Akurasi uji akhir': round(h['test_acc'][-1], 4),
    'Loss uji akhir': round(h['test_loss'][-1], 4),
    'Waktu latih (detik)': round(h['time_s'], 1)} for name, h in results.items()])
summary

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ep = range(1, EPOCHS + 1)
for name, h in results.items():
    ax1.plot(ep, h['train_loss'], '--', label=f'{name} (latih)')
    ax1.plot(ep, h['test_loss'], '-o', label=f'{name} (uji)')
    ax2.plot(ep, h['test_acc'], '-o', label=name)
ax1.set(title='Loss per epoch', xlabel='Epoch', ylabel='Cross-entropy'); ax1.legend(fontsize=8)
ax2.set(title='Akurasi uji per epoch', xlabel='Epoch', ylabel='Akurasi'); ax2.legend()
for ax in (ax1, ax2): ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Apa yang Dipelajari Jaringan?
### 7a. Filter conv1 (16 kernel 3×3×3)

In [ ]:
w = models['max_fc'].features[0].weight.detach().cpu()
w = (w - w.min()) / (w.max() - w.min())
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for i, ax in enumerate(axes.flat):
    ax.imshow(w[i].permute(1, 2, 0)); ax.set_title(f'f{i}', fontsize=8); ax.axis('off')
plt.suptitle('Filter conv1 (SmallCNN max_fc)'); plt.tight_layout(); plt.show()

### 7b. Feature map layer 1 dan layer 2 untuk satu citra uji

In [ ]:
model = models['max_fc'].eval()
x0, y0 = test_ds[0]
with torch.no_grad():
    fmap1 = model.features[:2](x0[None].to(DEVICE)).cpu()[0]    # setelah conv1 + ReLU
    fmap2 = model.features[:5](x0[None].to(DEVICE)).cpu()[0]    # setelah conv2 + ReLU

def show_maps(fm, title, n=8):
    fig, axes = plt.subplots(1, n, figsize=(14, 2))
    for i, ax in enumerate(axes):
        ax.imshow(fm[i], cmap='viridis'); ax.axis('off')
    plt.suptitle(title); plt.show()

im0 = x0 * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
plt.figure(figsize=(2, 2)); plt.imshow(im0.permute(1, 2, 0).clamp(0, 1)); plt.title(CLASSES[y0]); plt.axis('off'); plt.show()
show_maps(fmap1, 'Feature map setelah conv1 + ReLU (32×32)')
show_maps(fmap2, 'Feature map setelah conv2 + ReLU (16×16)')

### 7c. Akurasi per kelas

In [ ]:
_, _, preds, gts = evaluate(models['max_fc'], test_dl, nn.CrossEntropyLoss())
acc_cls = [(preds[gts == c] == c).float().mean().item() if (gts == c).any() else float('nan') for c in range(N_CLS)]
order = np.argsort(acc_cls)
plt.figure(figsize=(8, 0.4 * N_CLS + 1.5))
plt.barh([CLASSES[i] for i in order], [acc_cls[i] for i in order], color='#15596B')
plt.xlabel('Akurasi'); plt.title('Akurasi per kelas — SmallCNN max_fc'); plt.xlim(0, 1); plt.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Pertanyaan Analisis
Tulis jawaban pada sel markdown di bawah ini (masing-masing 1 paragraf).

1. Apakah jumlah parameter hasil `trace_model` sama dengan perhitungan manual di slide 16? Layer mana yang paling "mahal", dan mengapa?
2. Bandingkan `max_fc` dengan `avg_fc`. Mengapa jumlah parameternya identik, tetapi akurasinya bisa berbeda?
3. Varian `max_gap` hanya memakai ±2% parameter dari `max_fc`. Bagaimana akurasinya setelah 5 epoch? Apa yang kemungkinan terjadi bila pelatihan diperpanjang? Kaitkan dengan kapasitas model dan overfitting.
4. Amati filter conv1 dan feature map. Adakah yang menyerupai detektor tepi dari Minggu 3?
5. Kelas mana yang paling sulit dikenali? Berikan hipotesis berdasarkan kemiripan visual antarkelas.

**Jawaban:**

1. …
2. …
3. …
4. …
5. …